# SFT Fine-Tuning: `google/gemma-2-2b` on `badlogicgames/pi-mono`
### `fine-tuning-agent-on-traces-v2` | Google Colab Free Tier (T4 GPU)

This notebook fine-tunes **Gemma 2B** on coding-agent execution traces from `badlogicgames/pi-mono`.

- **Unsloth 4-bit QLoRA** (memory-efficient on T4)
- **TRL `SFTConfig.completion_only_loss=True`** (native completion-only loss)
- **BitsAndBytesConfig** with `bnb_4bit_compute_dtype=torch.float16` (fixes Half/BFloat16 dtype mismatch on T4)
- **TrackIO** for experiment tracking
- **Inspect AI** for HumanEval + MBPP benchmarks
- Adapters and merged weights pushed to Hugging Face Hub

## Cell 1: Install Dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q \
    "trl>=1.6.0" \
    "peft>=0.18.0" \
    "accelerate>=1.13.0" \
    "bitsandbytes>=0.48.0" \
    "transformers>=5.5.0" \
    "datasets>=4.4.0" \
    "huggingface_hub>=1.1.0" \
    "trackio>=0.3.0" \
    "inspect-ai"
print("✅ All dependencies installed.")

## Cell 2: Authentication & Configuration

In [ ]:
import os
from huggingface_hub import HfApi, login

MODEL_ID = "unsloth/gemma-2-2b-it-bnb-4bit"
DATASET_ID = "badlogicgames/pi-mono"
MAX_SEQ_LENGTH = 2048
TRACKIO_PROJECT = "agent-fine-tuning-on-trace-v2"
HUB_REPO_SUFFIX = "fine-tuning-agent-on-traces-v2"

SWEEP_CONFIGS = [
    {"job_id": "job_01_lr1e4_r16", "learning_rate": 1e-4, "lora_r": 16, "lora_alpha": 32},
    {"job_id": "job_02_lr2e4_r32", "learning_rate": 2e-4, "lora_r": 32, "lora_alpha": 64},
    {"job_id": "job_03_lr5e5_r16", "learning_rate": 5e-5, "lora_r": 16, "lora_alpha": 32},
]

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = input("Paste your Hugging Face Write token: ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

api = HfApi()
HF_USERNAME = api.whoami()['name']
FINAL_REPO_ID = f"{HF_USERNAME}/{HUB_REPO_SUFFIX}"

print(f"✅ Authenticated as: {HF_USERNAME}")
print(f"   Model:      {MODEL_ID}")
print(f"   Final Repo: https://huggingface.co/{FINAL_REPO_ID}")
print(f"   TrackIO:    {TRACKIO_PROJECT}")

## Cell 3: Download & Preprocess `badlogicgames/pi-mono` Traces

In [ ]:
import copy
import hashlib
import json
import random
from pathlib import Path
from typing import Any

from datasets import Dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

print("Downloading raw JSONL trace files from badlogicgames/pi-mono...")
raw_dir = Path("./pi_mono_raw")
snapshot_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    allow_patterns=["*.jsonl"],
    local_dir=str(raw_dir),
)
trace_files = sorted(raw_dir.glob("*.jsonl"))
print(f"✅ Downloaded {len(trace_files)} trace files.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def clip_text(text: str, max_chars: int = 12000) -> str:
    if max_chars <= 0 or len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head
    return f"{text[:head]}\n\n[... omitted {len(text) - max_chars} chars ...]\n\n{text[-tail:]}"

def extract_text_parts(parts: Any, max_chars: int = 12000) -> str:
    if isinstance(parts, str):
        return clip_text(parts.strip(), max_chars)
    if not isinstance(parts, list):
        return ""
    out = []
    for part in parts:
        if not isinstance(part, dict):
            continue
        ptype = part.get("type")
        if ptype == "text":
            v = str(part.get("text") or "").strip()
            if v:
                out.append(v)
        elif ptype == "image":
            out.append("[image omitted]")
    return clip_text("\n".join(out).strip(), max_chars)

def convert_tool_call(part: dict) -> dict | None:
    name = part.get("name")
    if not name:
        return None
    args = part.get("arguments") or {}
    call_id = str(part.get("id") or f"call_{hashlib.sha1(json.dumps(part, sort_keys=True, default=str).encode()).hexdigest()[:12]}")
    return {"id": call_id, "type": "function", "function": {"name": str(name), "arguments": args}}

def raw_event_to_chat_message(event: dict) -> dict | None:
    if event.get("type") != "message":
        return None
    raw = event.get("message") or {}
    role = raw.get("role")
    if role == "user":
        content = extract_text_parts(raw.get("content") or [])
        return {"role": "user", "content": content} if content else None
    if role == "assistant":
        parts = raw.get("content") or []
        text = extract_text_parts(parts)
        tool_calls = []
        if isinstance(parts, list):
            for p in parts:
                if isinstance(p, dict) and p.get("type") == "toolCall":
                    tc = convert_tool_call(p)
                    if tc:
                        tool_calls.append(tc)
        if not text and not tool_calls:
            return None
        msg = {"role": "assistant", "content": text}
        if tool_calls:
            msg["tool_calls"] = tool_calls
        return msg
    if role == "toolResult":
        content = extract_text_parts(raw.get("content") or [], max_chars=12000) or "[empty tool result]"
        return {"role": "tool", "tool_call_id": str(raw.get("toolCallId") or ""), "name": str(raw.get("toolName") or "unknown"), "content": content}
    return None

print("\nConverting traces to prompt/completion pairs...")
examples = []
for path in trace_files:
    events = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.strip():
                try:
                    events.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    conversation = []
    for event in events:
        msg = raw_event_to_chat_message(event)
        if msg is None:
            continue
        if msg["role"] == "assistant" and any(m["role"] == "user" for m in conversation):
            try:
                prompt = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
                full = tokenizer.apply_chat_template(conversation + [msg], tokenize=False, add_generation_prompt=False)
                if full.startswith(prompt) and full[len(prompt):].strip():
                    completion = full[len(prompt):]
                    total_len = len(tokenizer(prompt + completion, add_special_tokens=False)["input_ids"])
                    if total_len <= MAX_SEQ_LENGTH:
                        examples.append({"prompt": prompt, "completion": completion})
            except Exception:
                pass
        conversation.append(msg)

rng = random.Random(42)
rng.shuffle(examples)
eval_size = min(256, max(1, len(examples) // 20))
eval_examples = examples[:eval_size]
train_examples = examples[eval_size:]
train_ds = Dataset.from_list(train_examples)
eval_ds = Dataset.from_list(eval_examples)

print(f"✅ Built {len(examples)} examples total.")
print(f"   Train: {len(train_ds)} | Eval: {len(eval_ds)}")

## Cell 4: Hyperparameter Sweep Training Loop

In [ ]:
import gc
import torch
import trackio

from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

sweep_results = []

for cfg in SWEEP_CONFIGS:
    job_id = cfg["job_id"]
    print(f"\n{'='*50}")
    print(f"Starting Sweep Job: {job_id}")
    print(f"Model: {MODEL_ID}")
    print(f"LR={cfg['learning_rate']} | LoRA r={cfg['lora_r']} | alpha={cfg['lora_alpha']}")
    print(f"{'='*50}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.config.use_cache = False

    peft_config = LoraConfig(
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    trackio.init(
        project=TRACKIO_PROJECT,
        name=job_id,
        group="pi-mono-sft-sweep",
        config={
            "model": MODEL_ID,
            "dataset": DATASET_ID,
            "learning_rate": cfg["learning_rate"],
            "lora_r": cfg["lora_r"],
            "lora_alpha": cfg["lora_alpha"],
            "completion_only_loss": True,
        },
    )

    output_dir = f"./results/{job_id}"
    sft_config = SFTConfig(
        output_dir=output_dir,
        max_length=MAX_SEQ_LENGTH,
        completion_only_loss=True,
        packing=False,
        dataset_text_field=None,
        learning_rate=cfg["learning_rate"],
        max_steps=60,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=20,
        save_steps=60,
        save_total_limit=1,
        fp16=False,
        bf16=False,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        report_to="none",
        remove_unused_columns=True,
        seed=42,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        peft_config=peft_config,
        processing_class=tokenizer,
    )

    train_result = trainer.train()

    eval_entry = next(
        (row for row in reversed(trainer.state.log_history) if "eval_loss" in row),
        None,
    )
    held_out_eval_loss = eval_entry["eval_loss"] if eval_entry else float("inf")

    trackio.log({
        "held_out_eval_loss": held_out_eval_loss,
        "train_loss": train_result.training_loss,
        "train_steps": train_result.global_step,
    })
    trackio.finish()

    adapter_repo_id = f"{HF_USERNAME}/fine-tuning-agent-on-traces-v2-{job_id}"
    trainer.save_model(output_dir)
    trainer.model.push_to_hub(adapter_repo_id, token=HF_TOKEN)
    tokenizer.push_to_hub(adapter_repo_id, token=HF_TOKEN)

    sweep_results.append({
        "job_id": job_id,
        "params": cfg,
        "eval_loss": held_out_eval_loss,
        "adapter_repo_id": adapter_repo_id,
        "output_dir": output_dir,
    })

    print(f"\n✅ {job_id} DONE | eval_loss={held_out_eval_loss:.4f} | adapter -> {adapter_repo_id}")

    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*50)
print("ALL SWEEP JOBS COMPLETE")
for r in sweep_results:
    print(f"  {r['job_id']}: eval_loss={r['eval_loss']:.4f}")

## Cell 5: Select Best Run & Push Final Merged Weights

In [ ]:
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

best_run = min(sweep_results, key=lambda x: x["eval_loss"])
print(f"🏆 Best Run: {best_run['job_id']}")
print(f"   Eval Loss: {best_run['eval_loss']:.4f}")
print(f"   Params: LR={best_run['params']['learning_rate']} | r={best_run['params']['lora_r']} | alpha={best_run['params']['lora_alpha']}")

gc.collect()
torch.cuda.empty_cache()

print("\nLoading base model for merging (on CPU to avoid OOM)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
)
merge_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading adapter and merging...")
merged_model = PeftModel.from_pretrained(base_model, best_run["output_dir"])
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained("./final_merged_model")
merge_tokenizer.save_pretrained("./final_merged_model")

print(f"Pushing merged model to https://huggingface.co/{FINAL_REPO_ID} ...")
merged_model.push_to_hub(FINAL_REPO_ID, token=HF_TOKEN)
merge_tokenizer.push_to_hub(FINAL_REPO_ID, token=HF_TOKEN)
print(f"✅ Final merged model pushed to: https://huggingface.co/{FINAL_REPO_ID}")

del base_model, merged_model, merge_tokenizer
gc.collect()

## Cell 6: Inspect AI Benchmark Evaluations (HumanEval & MBPP)

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("Running HumanEval benchmark...")
!inspect eval humaneval --model hf/final_merged_model --log-dir ./inspect_logs

print("\nRunning MBPP benchmark...")
!inspect eval mbpp --model hf/final_merged_model --log-dir ./inspect_logs

print("\n✅ Benchmark evaluations complete. Check ./inspect_logs for results.")

## Cell 6.5: Before vs After — Model Behaviour Comparison
> Runs the **base model** and the **fine-tuned model** on the same coding-agent prompts side by side.
> Results are stored in `before_results` / `after_results` and automatically included in the README (Cell 7).

In [ ]:
import gc
import textwrap
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- Prompts representative of pi-mono coding-agent traces ---
EVAL_PROMPTS = [
    {
        "label": "List workspace files",
        "messages": [{"role": "user", "content": "List the files in the current workspace directory."}],
    },
    {
        "label": "Read a file",
        "messages": [{"role": "user", "content": "Read the file README.md and summarise its contents."}],
    },
    {
        "label": "Debug a function",
        "messages": [{"role": "user", "content": "The function `calculate_sum` returns the wrong result for negative numbers. Find the bug and fix it."}],
    },
    {
        "label": "Write a test",
        "messages": [{"role": "user", "content": "Write a pytest test for the `parse_config` function that reads a YAML file and returns a dict."}],
    },
]

def run_inference(model, tok, prompts, max_new_tokens=200):
    """Greedy inference on a list of prompt dicts. Returns list of (label, response)."""
    results = []
    model.eval()
    with torch.no_grad():
        for item in prompts:
            prompt_str = tok.apply_chat_template(
                item["messages"], tokenize=False, add_generation_prompt=True
            )
            inputs = tok(prompt_str, return_tensors="pt").to(model.device)
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tok.eos_token_id,
            )
            new_ids = out_ids[0][inputs["input_ids"].shape[-1]:]
            response = tok.decode(new_ids, skip_special_tokens=True).strip()
            results.append((item["label"], response))
    return results

def print_comparison(before, after=None):
    """Pretty-print before (and optionally after) responses."""
    W = 90
    SEP = "═" * W
    for i, (label, b_resp) in enumerate(before):
        print(f"\n{SEP}")
        print(f"  📝 PROMPT: {label}")
        print(SEP)
        print("  🔵 BEFORE (base model):")
        for line in textwrap.wrap(b_resp or "[no output]", W - 6):
            print(f"     {line}")
        if after:
            _, a_resp = after[i]
            print("  🟢 AFTER  (fine-tuned):")
            for line in textwrap.wrap(a_resp or "[no output]", W - 6):
                print(f"     {line}")
    print(f"\n{SEP}")

bnb_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ── 1. BASE MODEL ──────────────────────────────────────────────────────────────
print("[1/2] Running BASE model inference...")
gc.collect()
torch.cuda.empty_cache()

base_tok = AutoTokenizer.from_pretrained(MODEL_ID)
if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token
base_inf = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_inf, device_map="auto"
)
before_results = run_inference(base_inf, base_tok, EVAL_PROMPTS)

del base_inf, base_tok
gc.collect()
torch.cuda.empty_cache()

# ── 2. FINE-TUNED MODEL ────────────────────────────────────────────────────────
print("\n[2/2] Running FINE-TUNED model inference...")
ft_tok = AutoTokenizer.from_pretrained(MODEL_ID)
if ft_tok.pad_token is None:
    ft_tok.pad_token = ft_tok.eos_token
ft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_inf, device_map="auto"
)
ft_model = PeftModel.from_pretrained(ft_base, best_run["output_dir"])
after_results = run_inference(ft_model, ft_tok, EVAL_PROMPTS)

del ft_model, ft_base, ft_tok
gc.collect()
torch.cuda.empty_cache()

# ── 3. PRINT SIDE-BY-SIDE ──────────────────────────────────────────────────────
print("\n" + "═" * 90)
print("  BEFORE vs AFTER FINE-TUNING")
print("═" * 90)
print_comparison(before_results, after_results)

# ── 4. SWEEP SUMMARY ───────────────────────────────────────────────────────────
print("\n  SWEEP SUMMARY")
print("  " + "-" * 62)
print(f"  {'Job ID':<25} {'LR':<10} {'LoRA r':<8} {'Eval Loss'}")
print("  " + "-" * 62)
for r in sweep_results:
    marker = " ← BEST" if r['job_id'] == best_run['job_id'] else ""
    print(f"  {r['job_id']:<25} {str(r['params']['learning_rate']):<10} {str(r['params']['lora_r']):<8} {r['eval_loss']:.4f}{marker}")
print("  " + "-" * 62)
print(f"  Best adapter : https://huggingface.co/{best_run['adapter_repo_id']}")
print(f"  Final model  : https://huggingface.co/{FINAL_REPO_ID}")

print("\n✅ Comparison done. Proceed to Cell 7 to bake this into the README.")

## Cell 7: Generate & Push Model README.md to Hugging Face Hub

In [ ]:
import json
from pathlib import Path
from huggingface_hub import HfApi

def parse_inspect_score(log_dir: str, task_name: str) -> str:
    log_path = Path(log_dir)
    for f in sorted(log_path.glob("*.json"), reverse=True):
        try:
            data = json.loads(f.read_text())
            if data.get("eval", {}).get("task") == task_name:
                score = data.get("results", {}).get("metrics", {}).get("mean", {}).get("value")
                if score is not None:
                    return f"{score:.1%}"
        except Exception:
            pass
    return "*see inspect_logs/*"

humaneval_score = parse_inspect_score("./inspect_logs", "humaneval")
mbpp_score = parse_inspect_score("./inspect_logs", "mbpp")

# --- Before/After table rows (from Cell 6.5) ---
ba_rows = ""
for (label, b_resp), (_, a_resp) in zip(before_results, after_results):
    # Truncate to 250 chars, escape markdown pipes
    def fmt(s):
        s = s.replace("|", "\\|").replace("\n", " ").strip()
        return (s[:250] + "…") if len(s) > 250 else s
    ba_rows += f"| **{label}** | {fmt(b_resp)} | {fmt(a_resp)} |\n"

# --- Sweep table rows ---
sweep_rows = ""
for r in sweep_results:
    p = r["params"]
    marker = " 🏆" if r["job_id"] == best_run["job_id"] else ""
    sweep_rows += (
        f"| `{r['job_id']}`{marker} "
        f"| `{p['learning_rate']}` "
        f"| `{p['lora_r']}` "
        f"| `{p['lora_alpha']}` "
        f"| `{r['eval_loss']:.4f}` "
        f"| [{r['adapter_repo_id']}](https://huggingface.co/{r['adapter_repo_id']}) |\n"
    )

readme = f"""---
license: gemma
base_model: google/gemma-2-2b
datasets:
- badlogicgames/pi-mono
tags:
- sft
- lora
- coding-agent
---

# `{FINAL_REPO_ID}`

Fine-tuned **Gemma 2B** (`google/gemma-2-2b`) on coding-agent execution traces from
[`badlogicgames/pi-mono`](https://huggingface.co/datasets/badlogicgames/pi-mono)
using **4-bit QLoRA** and **completion-only loss**.

Methodology follows [`burtenshaw/training-agents`](https://github.com/burtenshaw/training-agents),
adapted for Google Colab Free Tier (Tesla T4 GPU) via Unsloth.

## 🔄 Before vs After Fine-Tuning

Greedy decoding, `max_new_tokens=200`, same prompts on base and fine-tuned model.

| Prompt | 🔵 Base Model | 🟢 Fine-tuned |
| :--- | :--- | :--- |
{ba_rows}
## 📊 Benchmark Results

Evaluated with [Inspect AI](https://inspect.ai-safety-institute.org.uk/).

| Benchmark | Metric | Score |
| :--- | :--- | :--- |
| HumanEval | pass@1 (temp=0.0) | {humaneval_score} |
| MBPP | pass@1 (temp=0.5) | {mbpp_score} |

## 🧪 Hyperparameter Sweep

TrackIO project: **`{TRACKIO_PROJECT}`** · Best run selected by lowest held-out eval loss.

| Job ID | LR | LoRA r | LoRA alpha | Eval Loss | Adapter |
| :--- | :--- | :--- | :--- | :--- | :--- |
{sweep_rows}
## 🔗 Links

- **Final model**: [{FINAL_REPO_ID}](https://huggingface.co/{FINAL_REPO_ID})
- **Dataset**: [badlogicgames/pi-mono](https://huggingface.co/datasets/badlogicgames/pi-mono)
- **Reference code**: [burtenshaw/training-agents](https://github.com/burtenshaw/training-agents)

## ⚠️ Known Limitations

1. Trained on coding-agent tool traces (`bash`, `read`, `write`, `edit`, `grep`). General tasks may regress slightly vs base.
2. Max sequence length: {MAX_SEQ_LENGTH} tokens. Longer traces were filtered before training.
3. 60 training steps per sweep job (Colab T4 constraint). More steps may further reduce eval loss.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

HfApi().upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=FINAL_REPO_ID,
    token=HF_TOKEN,
)
print("✅ README.md with before/after table pushed to Hub!")
print(f"   View at: https://huggingface.co/{FINAL_REPO_ID}")